# YOLOv1 object detector: training notebook (Kaggle & Colab)

Train the YOLOv1 detector with a **free Kaggle or Colab GPU** on synthetic shapes, MNIST
digits, PASCAL VOC or any dataset in the YOLO text format (for example the Mapillary
traffic-sign export). The notebook is **self-contained**: section 1 writes the `yolov1`
package (identical to the repo's `src/`), and training goes through the same `train()`
function as the `yolov1-train` command.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 x2* (or P100) | *Runtime → Change runtime type → T4 GPU* |
| Internet | *Settings → Internet → On* (to download data) | on by default |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

**Sections:** 1. The package · 2. Settings · 3. Data · 4. Model · 5. Training with a live
dashboard · 6. Results · 7. Download

## 1. The package

The cells below write `src/yolov1/*.py`, copied from the repo at build time. Read them in this
order: `config` → `boxes` (target encoding / decoding, NMS) → `model` → `loss` → `metrics` →
`data` → `train`.

In [ ]:
import os

os.makedirs("src/yolov1/data", exist_ok=True)

In [ ]:
%%writefile src/yolov1/__init__.py
"""YOLOv1 object detection in PyTorch: model, loss, box utilities, datasets, training and inference."""

from yolov1.boxes import box_iou, decode_predictions, encode_targets, non_max_suppression
from yolov1.config import DataConfig, ModelConfig, TrainConfig
from yolov1.loss import YoloLoss
from yolov1.metrics import mean_average_precision
from yolov1.model import YoloV1

__all__ = [
    "DataConfig",
    "ModelConfig",
    "TrainConfig",
    "YoloLoss",
    "YoloV1",
    "box_iou",
    "decode_predictions",
    "encode_targets",
    "mean_average_precision",
    "non_max_suppression",
]
__version__ = "1.0.0"

In [ ]:
%%writefile src/yolov1/config.py
"""All settings, grouped in three dataclasses. The CLI flags of yolov1-train map 1:1 to these fields."""

from dataclasses import dataclass


@dataclass
class DataConfig:
    dataset: str = "shapes"  # shapes | mnist | voc | yolo
    root: str = "data"  # download / dataset folder (mnist, voc, yolo)
    image_size: int = 448  # network input; must be divisible by the architecture stride (64 or 16)
    grid_size: int = 7  # S: the image is split in S x S cells
    num_boxes: int = 2  # B: boxes predicted per cell
    train_size: int = 2000  # shapes / mnist: generated training images
    val_size: int = 200  # shapes / mnist: generated validation images
    max_objects: int = 3  # shapes / mnist: objects per image (1..max_objects)
    val_fraction: float = 0.1  # yolo folder without a train/val split
    classes: list[str] | None = None  # yolo: class names (else classes.txt or "0", "1", ...)
    images_dir: str = "images"  # yolo: image folder, relative to root
    labels_dir: str = "labels"  # yolo: label folder, relative to root
    voc_year: str = "2012"  # voc: 2007 or 2012
    augment: bool = True  # flip, random scale/translate and color jitter on the training split
    num_workers: int = 2
    seed: int = 0


@dataclass
class ModelConfig:
    arch: str = "yolov1"  # yolov1 = the 24-conv paper network (stride 64) | small = 6 convs, stride 16
    width: float = 1.0  # channel multiplier (0.25 for quick CPU runs)
    hidden: int = 4096  # units of the first fully-connected layer (paper: 4096)
    dropout: float = 0.5  # paper: 0.5 after the first fully-connected layer


@dataclass
class TrainConfig:
    epochs: int = 50
    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 5e-4
    warmup_epochs: int = 1
    lambda_coord: float = 5.0  # paper weights
    lambda_noobj: float = 0.5
    grad_clip: float = 10.0
    conf_threshold: float = 0.1  # evaluation: keep boxes with class score above this before NMS
    nms_iou: float = 0.5  # evaluation: NMS IoU threshold
    map_iou: float = 0.5  # evaluation: a prediction is a true positive above this IoU
    amp: bool = True  # mixed precision on CUDA
    device: str = "auto"
    output_dir: str = "checkpoints/run"
    overfit: int = 0  # >0: train and validate on this many training samples (sanity check)
    log_every: int = 50
    plot_every: int = 1  # redraw history.png / predictions.png every N epochs (0: only at the end)
    seed: int = 0

In [ ]:
%%writefile src/yolov1/boxes.py
"""Box utilities: IoU, target encoding, prediction decoding and non-maximum suppression.

Box formats (all coordinates relative to the image, in [0, 1]):
  cxcywh  (center x, center y, width, height)  used by labels, targets and predictions
  xyxy    (x_min, y_min, x_max, y_max)          used for IoU and drawing

Detections are rows  [class, score, cx, cy, w, h];  ground truth rows are  [class, cx, cy, w, h].
"""

from __future__ import annotations

import torch


def cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), dim=-1)


def xyxy_to_cxcywh(boxes: torch.Tensor) -> torch.Tensor:
    x1, y1, x2, y2 = boxes.unbind(-1)
    return torch.stack(((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1), dim=-1)


def box_iou(a: torch.Tensor, b: torch.Tensor, box_format: str = "cxcywh", eps: float = 1e-6) -> torch.Tensor:
    """Element-wise IoU of two broadcastable (..., 4) tensors -> (...)."""
    if box_format == "cxcywh":
        a, b = cxcywh_to_xyxy(a), cxcywh_to_xyxy(b)
    elif box_format != "xyxy":
        raise ValueError(f"unknown box_format {box_format!r}")
    lt = torch.max(a[..., :2], b[..., :2])
    rb = torch.min(a[..., 2:], b[..., 2:])
    inter = (rb - lt).clamp(min=0).prod(-1)
    area_a = (a[..., 2:] - a[..., :2]).clamp(min=0).prod(-1)
    area_b = (b[..., 2:] - b[..., :2]).clamp(min=0).prod(-1)
    return inter / (area_a + area_b - inter + eps)


def pairwise_iou(a: torch.Tensor, b: torch.Tensor, box_format: str = "cxcywh") -> torch.Tensor:
    """IoU between every box of a (N, 4) and every box of b (M, 4) -> (N, M)."""
    return box_iou(a[:, None, :], b[None, :, :], box_format)


def encode_targets(boxes: torch.Tensor, grid_size: int, num_boxes: int, num_classes: int) -> torch.Tensor:
    """Ground truth boxes (n, 5) [class, cx, cy, w, h] -> target grid (S, S, C + 5B).

    The cell containing the box center gets: one-hot class, confidence 1 in box slot 0 and the box
    as (x offset in the cell, y offset in the cell, w * S, h * S), as in the paper. If several
    objects fall in one cell only the first is kept: YOLOv1 predicts one object per cell.
    """
    S, B, C = grid_size, num_boxes, num_classes
    target = torch.zeros(S, S, C + 5 * B)
    for cls, cx, cy, w, h in boxes.tolist():
        col, row = min(int(cx * S), S - 1), min(int(cy * S), S - 1)  # cx = 1.0 belongs to the last cell
        if target[row, col, C] == 1:
            continue
        target[row, col, int(cls)] = 1
        target[row, col, C] = 1
        target[row, col, C + 1 : C + 5] = torch.tensor([cx * S - col, cy * S - row, w * S, h * S])
    return target


def decode_predictions(pred: torch.Tensor, num_boxes: int, num_classes: int) -> list[torch.Tensor]:
    """Network output (N, S, S, C + 5B) -> one (S*S*B, 6) tensor of [class, score, cx, cy, w, h] per image.

    score = box confidence * best class probability (the paper's class-specific confidence), with
    everything clamped to valid ranges because the network outputs are unbounded.
    """
    N, S, _, _ = pred.shape
    B, C = num_boxes, num_classes
    pred = pred.detach().float()
    class_prob, class_idx = pred[..., :C].clamp(0, 1).max(-1)  # (N, S, S)
    boxes = pred[..., C:].reshape(N, S, S, B, 5)
    conf = boxes[..., 0].clamp(0, 1)  # (N, S, S, B)
    cols = torch.arange(S, device=pred.device, dtype=pred.dtype)
    cx = (boxes[..., 1] + cols.view(1, 1, S, 1)) / S
    cy = (boxes[..., 2] + cols.view(1, S, 1, 1)) / S
    wh = boxes[..., 3:5].clamp(min=0) / S
    score = conf * class_prob.unsqueeze(-1)
    cls = class_idx.unsqueeze(-1).expand_as(conf).to(pred.dtype)
    rows = torch.cat([cls.unsqueeze(-1), score.unsqueeze(-1), cx.unsqueeze(-1), cy.unsqueeze(-1), wh], -1)
    return [rows[i].reshape(-1, 6) for i in range(N)]


def non_max_suppression(detections: torch.Tensor, iou_threshold: float, score_threshold: float) -> torch.Tensor:
    """Greedy per-class NMS on (K, 6) [class, score, cx, cy, w, h] rows -> kept rows sorted by score."""
    detections = detections[detections[:, 1] > score_threshold]
    if detections.numel() == 0:
        return detections.reshape(0, 6)
    detections = detections[detections[:, 1].argsort(descending=True)]
    keep = []
    while detections.shape[0]:
        best, rest = detections[0], detections[1:]
        keep.append(best)
        if rest.shape[0] == 0:
            break
        same_class = rest[:, 0] == best[0]
        ious = box_iou(best[None, 2:6], rest[:, 2:6])
        detections = rest[~(same_class & (ious > iou_threshold))]
    return torch.stack(keep)

In [ ]:
%%writefile src/yolov1/model.py
"""The YOLOv1 network (Redmon et al., 2016, https://arxiv.org/abs/1506.02640).

A convolutional backbone (24 conv layers in the paper, "DarkNet") reduces the image by a factor
64, then two fully-connected layers output an S x S x (C + 5B) tensor: for each of the S x S grid
cells, C class probabilities and B boxes (confidence, cx, cy, w, h).

Each architecture is a list:  (out_channels, kernel, stride, padding) is a conv-bn-leaky block,
"M" is a 2x2 max-pool and [block, block, n] repeats the two blocks n times.
"""

from __future__ import annotations

import torch
from torch import nn

from yolov1.config import ModelConfig

Block = tuple[int, int, int, int]

# The paper network: 24 conv layers, 448 -> 7 (stride 64), 1024 output channels.
YOLOV1: list = [
    (64, 7, 2, 3),
    "M",
    (192, 3, 1, 1),
    "M",
    (128, 1, 1, 0),
    (256, 3, 1, 1),
    (256, 1, 1, 0),
    (512, 3, 1, 1),
    "M",
    [(256, 1, 1, 0), (512, 3, 1, 1), 4],
    (512, 1, 1, 0),
    (1024, 3, 1, 1),
    "M",
    [(512, 1, 1, 0), (1024, 3, 1, 1), 2],
    (1024, 3, 1, 1),
    (1024, 3, 2, 1),
    (1024, 3, 1, 1),
    (1024, 3, 1, 1),
]

# A small backbone for CPU runs and small images: 6 conv layers, stride 16 (112 -> 7).
SMALL: list = [
    (32, 3, 1, 1),
    "M",
    (64, 3, 1, 1),
    "M",
    (128, 3, 1, 1),
    "M",
    (256, 3, 1, 1),
    "M",
    (256, 3, 1, 1),
    (256, 3, 1, 1),
]

ARCHITECTURES = {"yolov1": YOLOV1, "small": SMALL}


class ConvBlock(nn.Module):
    """conv (no bias) -> batch norm -> leaky ReLU(0.1). The paper has no batch norm; it stabilises training."""

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int, padding: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


def _scaled(channels: int, width: float) -> int:
    return max(8, int(round(channels * width / 8)) * 8)


def build_backbone(arch: list, in_channels: int, width: float) -> tuple[nn.Sequential, int, int]:
    """Return (layers, out_channels, stride) for an architecture list."""
    layers: list[nn.Module] = []
    stride = 1

    def add(block: Block) -> None:
        nonlocal in_channels, stride
        out, k, s, p = block
        out = _scaled(out, width)
        layers.append(ConvBlock(in_channels, out, k, s, p))
        in_channels, stride = out, stride * s

    for item in arch:
        if isinstance(item, tuple):
            add(item)
        elif item == "M":
            layers.append(nn.MaxPool2d(2, 2))
            stride *= 2
        elif isinstance(item, list):
            first, second, repeats = item
            for _ in range(repeats):
                add(first)
                add(second)
        else:
            raise ValueError(f"unknown architecture item {item!r}")
    return nn.Sequential(*layers), in_channels, stride


class YoloV1(nn.Module):
    """YOLOv1 detector. forward(images) -> (N, S, S, C + 5B)."""

    def __init__(
        self,
        cfg: ModelConfig,
        num_classes: int,
        image_size: int = 448,
        grid_size: int = 7,
        num_boxes: int = 2,
        in_channels: int = 3,
    ):
        super().__init__()
        if cfg.arch not in ARCHITECTURES:
            raise ValueError(f"arch must be one of {sorted(ARCHITECTURES)}, got {cfg.arch!r}")
        self.num_classes, self.grid_size, self.num_boxes = num_classes, grid_size, num_boxes
        self.image_size = image_size
        self.backbone, channels, self.stride = build_backbone(ARCHITECTURES[cfg.arch], in_channels, cfg.width)
        if image_size % self.stride:
            raise ValueError(f"image_size {image_size} must be a multiple of the stride {self.stride}")
        feature = image_size // self.stride
        self.cell_values = num_classes + 5 * num_boxes
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels * feature * feature, cfg.hidden),
            nn.Dropout(cfg.dropout),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(cfg.hidden, grid_size * grid_size * self.cell_values),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.head(self.backbone(x))
        return out.view(-1, self.grid_size, self.grid_size, self.cell_values)

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

In [ ]:
%%writefile src/yolov1/loss.py
"""The YOLOv1 loss (paper, eq. 3): sum-squared error over cells and boxes.

For each cell with an object, the predictor whose box has the highest IoU with the ground truth is
"responsible" for it. The loss is

  lambda_coord * sum_resp [(x - x̂)² + (y - ŷ)² + (√w - √ŵ)² + (√h - √ĥ)²]      coordinates
  +              sum_resp (1 - Ĉ)²                                              confidence, responsible boxes
  + lambda_noobj * sum_not_resp Ĉ²                                              confidence, every other box
  +              sum_obj_cells sum_c (p(c) - p̂(c))²                             class probabilities

all divided by the batch size, so the learning rate does not depend on the batch size.
"""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn

from yolov1.boxes import box_iou


class YoloLoss(nn.Module):
    def __init__(
        self,
        grid_size: int,
        num_boxes: int,
        num_classes: int,
        lambda_coord: float = 5.0,
        lambda_noobj: float = 0.5,
    ):
        super().__init__()
        self.S, self.B, self.C = grid_size, num_boxes, num_classes
        self.lambda_coord, self.lambda_noobj = lambda_coord, lambda_noobj
        self.last_components: dict[str, float] = {}

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        N, S, B, C = pred.shape[0], self.S, self.B, self.C
        pred = pred.float().reshape(N, S, S, C + 5 * B)
        target = target.float()
        obj = target[..., C]  # (N, S, S) 1 where the cell holds an object
        tgt_box = target[..., C + 1 : C + 5]  # (N, S, S, 4) cell-relative x, y and grid-relative w, h
        pred_boxes = pred[..., C:].reshape(N, S, S, B, 5)
        pred_conf, pred_xy, pred_wh = pred_boxes[..., 0], pred_boxes[..., 1:3], pred_boxes[..., 3:5]

        # Which of the B boxes is responsible: highest IoU with the ground truth box of its cell.
        with torch.no_grad():
            ious = box_iou(pred_boxes[..., 1:5], tgt_box.unsqueeze(3))  # (N, S, S, B)
            responsible = F.one_hot(ious.argmax(-1), B).float() * obj.unsqueeze(-1)  # (N, S, S, B)

        tgt_xy, tgt_wh = tgt_box[..., :2].unsqueeze(3), tgt_box[..., 2:4].unsqueeze(3)
        xy_err = ((pred_xy - tgt_xy) ** 2).sum(-1)
        sqrt_pred_wh = torch.sign(pred_wh) * torch.sqrt(pred_wh.abs() + 1e-6)
        wh_err = ((sqrt_pred_wh - torch.sqrt(tgt_wh)) ** 2).sum(-1)
        coord_loss = (responsible * (xy_err + wh_err)).sum()
        obj_loss = (responsible * (pred_conf - 1) ** 2).sum()
        noobj_loss = ((1 - responsible) * pred_conf**2).sum()
        class_loss = (obj.unsqueeze(-1) * (pred[..., :C] - target[..., :C]) ** 2).sum()

        total = (self.lambda_coord * coord_loss + obj_loss + self.lambda_noobj * noobj_loss + class_loss) / N
        self.last_components = {
            "coord": coord_loss.item() / N,
            "obj": obj_loss.item() / N,
            "noobj": noobj_loss.item() / N,
            "class": class_loss.item() / N,
        }
        return total

In [ ]:
%%writefile src/yolov1/metrics.py
"""Mean average precision (mAP) at a fixed IoU threshold, VOC style with all-point interpolation."""

from __future__ import annotations

import torch

from yolov1.boxes import pairwise_iou


def average_precision(recall: torch.Tensor, precision: torch.Tensor) -> float:
    """Area under the precision envelope of a (recall, precision) curve, both sorted by score."""
    r = torch.cat([torch.zeros(1), recall, torch.ones(1)])
    p = torch.cat([torch.zeros(1), precision, torch.zeros(1)])
    for i in range(p.numel() - 2, -1, -1):  # make precision monotonically decreasing
        p[i] = torch.max(p[i], p[i + 1])
    changes = torch.nonzero(r[1:] != r[:-1]).flatten()
    return float(((r[changes + 1] - r[changes]) * p[changes + 1]).sum())


def mean_average_precision(
    predictions: list[torch.Tensor],
    targets: list[torch.Tensor],
    num_classes: int,
    iou_threshold: float = 0.5,
) -> dict:
    """predictions[i]: (K_i, 6) [class, score, cx, cy, w, h]; targets[i]: (M_i, 5) [class, cx, cy, w, h].

    Returns {"map": float, "ap": {class_id: float}} over the classes that have at least one ground
    truth box. Each ground truth box can be matched once; extra detections are false positives.
    """
    assert len(predictions) == len(targets), "one prediction tensor per image"
    aps: dict[int, float] = {}
    for c in range(num_classes):
        gts = {i: t[t[:, 0] == c, 1:5] for i, t in enumerate(targets) if (t[:, 0] == c).any()}
        n_gt = sum(g.shape[0] for g in gts.values())
        if n_gt == 0:
            continue
        dets = [(i, row) for i, p in enumerate(predictions) for row in p[p[:, 0] == c]]
        if not dets:
            aps[c] = 0.0
            continue
        dets.sort(key=lambda d: -float(d[1][1]))
        matched = {i: torch.zeros(g.shape[0], dtype=torch.bool) for i, g in gts.items()}
        tp = torch.zeros(len(dets))
        for k, (img, row) in enumerate(dets):
            if img not in gts:
                continue
            ious = pairwise_iou(row[None, 2:6], gts[img])[0]
            best = int(ious.argmax())
            if ious[best] >= iou_threshold and not matched[img][best]:
                matched[img][best] = True
                tp[k] = 1
        tp_cum = tp.cumsum(0)
        fp_cum = (1 - tp).cumsum(0)
        aps[c] = average_precision(tp_cum / n_gt, tp_cum / (tp_cum + fp_cum))
    return {"map": sum(aps.values()) / len(aps) if aps else 0.0, "ap": aps}

In [ ]:
%%writefile src/yolov1/viz.py
"""Drawing boxes on images, prediction grids and training curves."""

from __future__ import annotations

import colorsys
from pathlib import Path

import matplotlib
import torch
from PIL import Image, ImageDraw, ImageFont

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from yolov1.data.transforms import denormalize  # noqa: E402


def class_color(index: int, total: int) -> tuple[int, int, int]:
    r, g, b = colorsys.hsv_to_rgb((index / max(total, 1)) % 1.0, 0.85, 0.95)
    return int(r * 255), int(g * 255), int(b * 255)


def draw_boxes(
    image: Image.Image,
    boxes: torch.Tensor,
    class_names: list[str],
    with_scores: bool = True,
    width: int = 2,
) -> Image.Image:
    """Draw [class, score, cx, cy, w, h] (or [class, cx, cy, w, h] when with_scores=False) rows on a copy."""
    image = image.copy().convert("RGB")
    draw = ImageDraw.Draw(image)
    W, H = image.size
    font = ImageFont.load_default()
    for row in boxes.tolist():
        if with_scores:
            cls, score, cx, cy, w, h = row
            label = f"{class_names[int(cls)]} {score:.2f}"
        else:
            cls, cx, cy, w, h = row
            label = class_names[int(cls)]
        color = class_color(int(cls), len(class_names))
        x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H  # center -> top-left corner
        x2, y2 = (cx + w / 2) * W, (cy + h / 2) * H
        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        tw, th = draw.textbbox((0, 0), label, font=font)[2:]
        ty = y1 - th - 2 if y1 - th - 2 > 0 else y1
        draw.rectangle([x1, ty, x1 + tw + 4, ty + th + 2], fill=color)
        draw.text((x1 + 2, ty), label, fill="black", font=font)
    return image


def tensor_to_pil(pixels: torch.Tensor) -> Image.Image:
    return Image.fromarray((denormalize(pixels).permute(1, 2, 0).cpu().numpy() * 255).astype("uint8"))


def plot_predictions(
    images: list[torch.Tensor],
    predictions: list[torch.Tensor],
    targets: list[torch.Tensor],
    class_names: list[str],
    path: str | Path,
    max_images: int = 8,
) -> None:
    n = min(max_images, len(images))
    fig, axes = plt.subplots(2, n, figsize=(2.6 * n, 5.6), squeeze=False)
    for i in range(n):
        pil = tensor_to_pil(images[i])
        axes[0, i].imshow(draw_boxes(pil, targets[i], class_names, with_scores=False))
        axes[1, i].imshow(draw_boxes(pil, predictions[i], class_names))
        axes[0, i].set_title("ground truth", fontsize=9)
        axes[1, i].set_title(f"prediction ({len(predictions[i])} boxes)", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def plot_history(history: list[dict], path: str | Path) -> None:
    epochs = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    axes[0].plot(epochs, [h["val_loss"] for h in history], label="validation")
    axes[0].set_title("YOLO loss")
    axes[0].legend()
    axes[1].plot(epochs, [h["val_map"] for h in history], "o-", color="tab:green")
    axes[1].set_title("validation mAP@0.5")
    axes[1].set_ylim(0, 1)
    for key in ("coord", "obj", "noobj", "class"):
        axes[2].plot(epochs, [h["train_" + key] for h in history], label=key)
    axes[2].set_title("train loss components")
    axes[2].legend()
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/yolov1/data/transforms.py
"""Image + box transforms. Every transform takes (PIL image, boxes (n, 5) [class, cx, cy, w, h]) and
returns the same pair, with box coordinates still relative to the (possibly cropped) image."""

from __future__ import annotations

import random

import torch
from PIL import Image
from torchvision import transforms as T
from torchvision.transforms import functional as TF

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, boxes):
        for t in self.transforms:
            image, boxes = t(image, boxes)
        return image, boxes


class Resize:
    """Square resize (no aspect ratio preservation, like the paper). Relative boxes do not change."""

    def __init__(self, size: int):
        self.size = size

    def __call__(self, image, boxes):
        return image.resize((self.size, self.size), Image.BILINEAR), boxes


class RandomHorizontalFlip:
    def __init__(self, p: float = 0.5):
        self.p = p

    def __call__(self, image, boxes):
        if random.random() < self.p:
            image = TF.hflip(image)
            boxes = boxes.clone()
            boxes[:, 1] = 1 - boxes[:, 1]
        return image, boxes


class RandomScaleTranslate:
    """Random crop / zoom-out of up to `amount` of the image size (paper: 20%), boxes clipped to the view.
    Boxes that keep less than `min_visible` of their area are dropped."""

    def __init__(self, amount: float = 0.2, min_visible: float = 0.3):
        self.amount, self.min_visible = amount, min_visible

    def __call__(self, image, boxes):
        w, h = image.size
        scale = 1 + random.uniform(-self.amount, self.amount)
        new_w, new_h = w * scale, h * scale
        dx = random.uniform(-self.amount, self.amount) * w
        dy = random.uniform(-self.amount, self.amount) * h
        # view window in the original image, in pixels
        x0, y0 = (w - new_w) / 2 + dx, (h - new_h) / 2 + dy
        image = image.transform(
            (int(round(new_w)), int(round(new_h))),
            Image.AFFINE,
            (1, 0, x0, 0, 1, y0),  # output (u, v) samples input (u + x0, v + y0)
            Image.BILINEAR,
            fillcolor=(124, 116, 104),
        )
        if boxes.shape[0] == 0:
            return image, boxes
        b = boxes.clone()
        x1 = (b[:, 1] - b[:, 3] / 2) * w - x0
        y1 = (b[:, 2] - b[:, 4] / 2) * h - y0
        x2 = (b[:, 1] + b[:, 3] / 2) * w - x0
        y2 = (b[:, 2] + b[:, 4] / 2) * h - y0
        area = (x2 - x1) * (y2 - y1)
        cx1, cy1 = x1.clamp(0, new_w), y1.clamp(0, new_h)
        cx2, cy2 = x2.clamp(0, new_w), y2.clamp(0, new_h)
        visible = (cx2 - cx1).clamp(min=0) * (cy2 - cy1).clamp(min=0)
        keep = visible > self.min_visible * area
        b = torch.stack(
            [b[:, 0], (cx1 + cx2) / 2 / new_w, (cy1 + cy2) / 2 / new_h, (cx2 - cx1) / new_w, (cy2 - cy1) / new_h], 1
        )
        return image, b[keep]


class ColorJitter:
    """Brightness / saturation jitter (paper: up to a factor 1.5), boxes untouched."""

    def __init__(self, factor: float = 0.5):
        self.jitter = T.ColorJitter(brightness=factor, saturation=factor, hue=0.02)

    def __call__(self, image, boxes):
        return self.jitter(image), boxes


class ToTensor:
    def __call__(self, image, boxes):
        return TF.normalize(TF.to_tensor(image), MEAN, STD), boxes


def train_transforms(image_size: int, augment: bool = True) -> Compose:
    steps = []
    if augment:
        steps += [RandomScaleTranslate(), RandomHorizontalFlip(), ColorJitter()]
    return Compose(steps + [Resize(image_size), ToTensor()])


def eval_transforms(image_size: int) -> Compose:
    return Compose([Resize(image_size), ToTensor()])


def denormalize(pixels: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(MEAN, device=pixels.device).view(3, 1, 1)
    std = torch.tensor(STD, device=pixels.device).view(3, 1, 1)
    return (pixels * std + mean).clamp(0, 1)

In [ ]:
%%writefile src/yolov1/data/shapes.py
"""Synthetic detection data: colored circles, squares and triangles on a plain background.

No download, fully deterministic (image i is always the same for a given seed), with exact boxes:
the ideal first dataset to prove the detector learns.
"""

from __future__ import annotations

import math
import random

import torch
from PIL import Image, ImageDraw
from torch.utils.data import Dataset

CLASSES = ["circle", "square", "triangle"]


def _draw_shape(draw: ImageDraw.ImageDraw, cls: int, cx: float, cy: float, r: float, color) -> None:
    if cls == 0:
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)
    elif cls == 1:
        draw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=color)
    else:
        pts = [(cx + r * math.cos(a), cy + r * math.sin(a)) for a in (-math.pi / 2, math.pi / 6, 5 * math.pi / 6)]
        draw.polygon(pts, fill=color)


class ShapesDetection(Dataset):
    """Returns (PIL image, boxes (n, 5) [class, cx, cy, w, h] relative to the image)."""

    def __init__(self, size: int, image_size: int = 128, max_objects: int = 3, seed: int = 0, split: str = "train"):
        self.size, self.image_size, self.max_objects = size, image_size, max_objects
        self.seed = seed * 1_000_003 + {"train": 0, "val": 1, "test": 2}[split] * 100_003
        self.class_names = CLASSES

    def __len__(self) -> int:
        return self.size

    def __getitem__(self, index: int):
        rng = random.Random(self.seed + index)
        s = self.image_size
        bg = tuple(rng.randint(150, 255) for _ in range(3))
        image = Image.new("RGB", (s, s), bg)
        draw = ImageDraw.Draw(image)
        boxes = []
        for _ in range(rng.randint(1, self.max_objects)):
            cls = rng.randrange(len(CLASSES))
            r = rng.uniform(0.08, 0.2) * s
            cx, cy = rng.uniform(r, s - r), rng.uniform(r, s - r)
            if any(
                abs(cx - bx * s) < (r + bw * s / 2) and abs(cy - by * s) < (r + bh * s / 2)
                for _, bx, by, bw, bh in boxes
            ):
                continue  # skip overlapping shapes: one object per cell is YOLOv1's assumption anyway
            color = tuple(rng.randint(0, 120) for _ in range(3))
            _draw_shape(draw, cls, cx, cy, r, color)
            if cls == 2:  # the triangle's tight box
                top, bottom = cy - r, cy + r * math.sin(math.pi / 6)
                half_w = r * math.cos(math.pi / 6)
                boxes.append([cls, cx / s, (top + bottom) / 2 / s, 2 * half_w / s, (bottom - top) / s])
            else:
                boxes.append([cls, cx / s, cy / s, 2 * r / s, 2 * r / s])
        return image, torch.tensor(boxes, dtype=torch.float32).reshape(-1, 5)

In [ ]:
%%writefile src/yolov1/data/mnist.py
"""MNIST detection: 1-3 handwritten digits pasted at random positions and sizes on a dark canvas.

Ten classes with real (noisy, varied) pixels but exact boxes. MNIST is downloaded by torchvision
(about 12 MB) the first time.
"""

from __future__ import annotations

import random

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import datasets

CLASSES = [str(d) for d in range(10)]


class MnistDetection(Dataset):
    def __init__(
        self,
        root: str,
        size: int,
        image_size: int = 112,
        max_objects: int = 3,
        seed: int = 0,
        split: str = "train",
        download: bool = True,
    ):
        self.mnist = datasets.MNIST(root, train=split == "train", download=download)
        self.size, self.image_size, self.max_objects = size, image_size, max_objects
        self.seed = seed * 1_000_003 + {"train": 0, "val": 1, "test": 2}[split] * 100_003
        self.class_names = CLASSES

    def __len__(self) -> int:
        return self.size

    def __getitem__(self, index: int):
        rng = random.Random(self.seed + index)
        s = self.image_size
        canvas = np.zeros((s, s), dtype=np.uint8)
        boxes = []
        for _ in range(rng.randint(1, self.max_objects)):
            digit, label = self.mnist[rng.randrange(len(self.mnist))]
            side = int(rng.uniform(0.25, 0.45) * s)
            digit = np.array(digit.resize((side, side), Image.BILINEAR))
            ys, xs = np.nonzero(digit > 50)
            if len(xs) == 0:
                continue
            x0, y0 = rng.randint(0, s - side), rng.randint(0, s - side)
            bx1, by1, bx2, by2 = x0 + xs.min(), y0 + ys.min(), x0 + xs.max() + 1, y0 + ys.max() + 1
            if any(
                bx1 < ox2 and bx2 > ox1 and by1 < oy2 and by2 > oy1 for ox1, oy1, ox2, oy2 in (b[5:] for b in boxes)
            ):
                continue  # keep digits apart
            region = canvas[y0 : y0 + side, x0 : x0 + side]
            np.maximum(region, digit, out=region)
            boxes.append(
                [label, (bx1 + bx2) / 2 / s, (by1 + by2) / 2 / s, (bx2 - bx1) / s, (by2 - by1) / s, bx1, by1, bx2, by2]
            )
        image = Image.fromarray(canvas).convert("RGB")
        return image, torch.tensor([b[:5] for b in boxes], dtype=torch.float32).reshape(-1, 5)

In [ ]:
%%writefile src/yolov1/data/voc.py
"""PASCAL VOC detection (the benchmark of the YOLOv1 paper), downloaded by torchvision (~2 GB for 2012)."""

from __future__ import annotations

import torch
from torch.utils.data import Dataset
from torchvision import datasets

CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]  # fmt: skip


class VocDetection(Dataset):
    def __init__(self, root: str, year: str = "2012", split: str = "train", download: bool = True):
        self.voc = datasets.VOCDetection(root, year=year, image_set=split, download=download)
        self.class_names = CLASSES

    def __len__(self) -> int:
        return len(self.voc)

    def __getitem__(self, index: int):
        image, annotation = self.voc[index]
        image = image.convert("RGB")
        w, h = image.size
        objects = annotation["annotation"]["object"]
        if isinstance(objects, dict):
            objects = [objects]
        rows = []
        for obj in objects:
            if obj.get("difficult", "0") == "1":
                continue
            b = obj["bndbox"]
            x1, y1, x2, y2 = (float(b[k]) for k in ("xmin", "ymin", "xmax", "ymax"))
            rows.append(
                [CLASSES.index(obj["name"]), (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h]
            )
        return image, torch.tensor(rows, dtype=torch.float32).reshape(-1, 5)

In [ ]:
%%writefile src/yolov1/data/yolo_txt.py
"""Any dataset in the YOLO text format (the Mapillary traffic-sign export used in 2023, Roboflow exports...).

    root/
      images/            name.jpg (or .png, .jpeg)       root/images/train, root/images/val
      labels/            name.txt: "class cx cy w h"     root/labels/train, root/labels/val
      classes.txt        optional, one class name per line

With `images/train` + `images/val` folders the split is used as is. Otherwise the files are split
by a seeded shuffle into disjoint train / validation sets (the 2023 notebook's "test" and
"validation" CSVs were subsets of the training CSV; see legacy/README.md).
"""

from __future__ import annotations

import random
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset

IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".bmp")


def read_label_file(path: Path) -> torch.Tensor:
    rows = []
    if path.exists():
        for line in path.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                rows.append([float(parts[0])] + [float(v) for v in parts[1:5]])
    return torch.tensor(rows, dtype=torch.float32).reshape(-1, 5)


def list_images(folder: Path) -> list[Path]:
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)


class YoloTxtDetection(Dataset):
    def __init__(self, images: list[Path], labels_dir: Path, class_names: list[str]):
        self.images, self.labels_dir, self.class_names = images, labels_dir, class_names

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int):
        path = self.images[index]
        image = Image.open(path).convert("RGB")
        boxes = read_label_file(self.labels_dir / f"{path.stem}.txt")
        boxes[:, 1:] = boxes[:, 1:].clamp(0, 1)
        return image, boxes


def load_class_names(root: Path, classes: list[str] | None, labels_dirs: list[Path]) -> list[str]:
    if classes:
        return list(classes)
    for name in ("classes.txt", "classes.names", "obj.names"):
        if (root / name).exists():
            return [line.strip() for line in (root / name).read_text().splitlines() if line.strip()]
    highest = -1
    for folder in labels_dirs:
        for txt in folder.glob("*.txt"):
            rows = read_label_file(txt)
            if rows.shape[0]:
                highest = max(highest, int(rows[:, 0].max()))
    return [str(i) for i in range(highest + 1)]


def build_yolo_txt(
    root: str | Path,
    images_dir: str = "images",
    labels_dir: str = "labels",
    classes: list[str] | None = None,
    val_fraction: float = 0.1,
    seed: int = 0,
) -> tuple[YoloTxtDetection, YoloTxtDetection]:
    root = Path(root)
    img_root, lbl_root = root / images_dir, root / labels_dir
    if (img_root / "train").is_dir() and (img_root / "val").is_dir():
        names = load_class_names(root, classes, [lbl_root / "train", lbl_root / "val"])
        return (
            YoloTxtDetection(list_images(img_root / "train"), lbl_root / "train", names),
            YoloTxtDetection(list_images(img_root / "val"), lbl_root / "val", names),
        )
    images = list_images(img_root)
    if not images:
        raise FileNotFoundError(f"no images found in {img_root}")
    random.Random(seed).shuffle(images)
    n_val = max(1, int(round(val_fraction * len(images)))) if len(images) > 1 else 0
    names = load_class_names(root, classes, [lbl_root])
    return YoloTxtDetection(images[n_val:], lbl_root, names), YoloTxtDetection(images[:n_val], lbl_root, names)

In [ ]:
%%writefile src/yolov1/data/__init__.py
"""Datasets. Every source yields (PIL image, boxes (n, 5) [class, cx, cy, w, h] relative to the image);
`DetectionDataset` applies the transforms and encodes the YOLO target grid."""

from __future__ import annotations

from dataclasses import dataclass

import torch
from torch.utils.data import Dataset, Subset

from yolov1.boxes import encode_targets
from yolov1.config import DataConfig
from yolov1.data.transforms import Compose, denormalize, eval_transforms, train_transforms

__all__ = ["DetectionDataset", "Datasets", "build_datasets", "collate_fn", "denormalize"]


class DetectionDataset(Dataset):
    """(image tensor (3, H, W), target grid (S, S, C + 5B), boxes (n, 5)) for every sample of `source`."""

    def __init__(self, source: Dataset, transforms: Compose, grid_size: int, num_boxes: int, num_classes: int):
        self.source, self.transforms = source, transforms
        self.grid_size, self.num_boxes, self.num_classes = grid_size, num_boxes, num_classes

    def __len__(self) -> int:
        return len(self.source)

    def __getitem__(self, index: int):
        image, boxes = self.source[index]
        pixels, boxes = self.transforms(image, boxes)
        target = encode_targets(boxes, self.grid_size, self.num_boxes, self.num_classes)
        return pixels, target, boxes


def collate_fn(batch):
    pixels, targets, boxes = zip(*batch, strict=True)
    return torch.stack(pixels), torch.stack(targets), list(boxes)


@dataclass
class Datasets:
    train: DetectionDataset
    val: DetectionDataset
    class_names: list[str]


def build_sources(cfg: DataConfig):
    """Return (train_source, val_source, class_names) for cfg.dataset."""
    if cfg.dataset == "shapes":
        from yolov1.data.shapes import ShapesDetection

        kw = dict(image_size=max(cfg.image_size, 64), max_objects=cfg.max_objects, seed=cfg.seed)
        train = ShapesDetection(cfg.train_size, split="train", **kw)
        val = ShapesDetection(cfg.val_size, split="val", **kw)
    elif cfg.dataset == "mnist":
        from yolov1.data.mnist import MnistDetection

        kw = dict(root=cfg.root, image_size=cfg.image_size, max_objects=cfg.max_objects, seed=cfg.seed)
        train = MnistDetection(size=cfg.train_size, split="train", **kw)
        val = MnistDetection(size=cfg.val_size, split="val", **kw)
    elif cfg.dataset == "voc":
        from yolov1.data.voc import VocDetection

        train = VocDetection(cfg.root, cfg.voc_year, "train")
        val = VocDetection(cfg.root, cfg.voc_year, "val")
    elif cfg.dataset == "yolo":
        from yolov1.data.yolo_txt import build_yolo_txt

        train, val = build_yolo_txt(cfg.root, cfg.images_dir, cfg.labels_dir, cfg.classes, cfg.val_fraction, cfg.seed)
    else:
        raise ValueError(f"unknown dataset {cfg.dataset!r} (shapes | mnist | voc | yolo)")
    return train, val, list(train.class_names)


def build_datasets(cfg: DataConfig, overfit: int = 0) -> Datasets:
    train_src, val_src, class_names = build_sources(cfg)
    if overfit:
        train_src = Subset(train_src, list(range(min(overfit, len(train_src)))))
        val_src = train_src
    C = len(class_names)
    train = DetectionDataset(
        train_src, train_transforms(cfg.image_size, cfg.augment and not overfit), cfg.grid_size, cfg.num_boxes, C
    )
    val = DetectionDataset(val_src, eval_transforms(cfg.image_size), cfg.grid_size, cfg.num_boxes, C)
    return Datasets(train, val, class_names)

In [ ]:
%%writefile src/yolov1/train.py
"""Training loop and the `yolov1-train` command.

Writes to --output-dir:  best.pt (highest validation mAP), last.pt (for --resume), history.json,
history.png (curves) and predictions.png (validation images with ground truth and predictions).
"""

from __future__ import annotations

import argparse
import dataclasses
import json
import logging
import math
import random
import time
from collections.abc import Callable
from dataclasses import dataclass, fields
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from yolov1.boxes import decode_predictions, non_max_suppression
from yolov1.config import DataConfig, ModelConfig, TrainConfig
from yolov1.data import Datasets, build_datasets, collate_fn
from yolov1.loss import YoloLoss
from yolov1.metrics import mean_average_precision
from yolov1.model import YoloV1
from yolov1.viz import plot_history, plot_predictions

log = logging.getLogger("yolov1")
BEST_CHECKPOINT, LAST_CHECKPOINT = "best.pt", "last.pt"


@dataclass
class DatasetInfo:
    class_names: list[str]
    image_size: int
    grid_size: int
    num_boxes: int

    @property
    def num_classes(self) -> int:
        return len(self.class_names)


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def lr_lambda(step: int, warmup_steps: int, total_steps: int) -> float:
    """Linear warmup then cosine decay to 1% of the peak."""
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))


def build_model(cfg: ModelConfig, info: DatasetInfo) -> YoloV1:
    return YoloV1(cfg, info.num_classes, info.image_size, info.grid_size, info.num_boxes)


def build_dataloaders(datasets: Datasets, cfg: TrainConfig, data_cfg: DataConfig, device: torch.device):
    kw = dict(
        batch_size=cfg.batch_size,
        num_workers=data_cfg.num_workers,
        pin_memory=device.type == "cuda",
        collate_fn=collate_fn,
        persistent_workers=data_cfg.num_workers > 0,
    )
    train_loader = DataLoader(datasets.train, shuffle=True, drop_last=len(datasets.train) > cfg.batch_size, **kw)
    val_loader = DataLoader(datasets.val, shuffle=False, **kw)
    return train_loader, val_loader


def postprocess(output: torch.Tensor, info: DatasetInfo, conf_threshold: float, nms_iou: float) -> list[torch.Tensor]:
    """Network output -> list of NMS-filtered detections (K, 6) per image, on the CPU."""
    return [
        non_max_suppression(d.cpu(), nms_iou, conf_threshold)
        for d in decode_predictions(output, info.num_boxes, info.num_classes)
    ]


@torch.no_grad()
def evaluate(model, loader, loss_fn, info: DatasetInfo, cfg: TrainConfig, device, keep_examples: int = 8) -> dict:
    model.eval()
    total_loss, n_batches = 0.0, 0
    predictions, targets = [], []
    examples = {"images": [], "predictions": [], "targets": []}
    for pixels, grid, boxes in loader:
        pixels, grid = pixels.to(device, non_blocking=True), grid.to(device, non_blocking=True)
        output = model(pixels)
        total_loss += loss_fn(output, grid).item()
        n_batches += 1
        detections = postprocess(output, info, cfg.conf_threshold, cfg.nms_iou)
        predictions += detections
        targets += [b.cpu() for b in boxes]
        if len(examples["images"]) < keep_examples:
            take = keep_examples - len(examples["images"])
            examples["images"] += list(pixels[:take].cpu())
            examples["predictions"] += detections[:take]
            examples["targets"] += [b.cpu() for b in boxes[:take]]
    model.train()
    result = mean_average_precision(predictions, targets, info.num_classes, cfg.map_iou)
    return {
        "loss": total_loss / max(1, n_batches),
        "map": result["map"],
        "ap": {info.class_names[c]: ap for c, ap in result["ap"].items()},
        "examples": examples,
    }


def save_checkpoint(path: Path, model, optimizer, scheduler, epoch: int, best_map: float, configs: dict, info) -> None:
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "best_map": best_map,
            "configs": configs,
            "info": dataclasses.asdict(info),
        },
        path,
    )


def load_model(path: str | Path, device: str | torch.device = "cpu") -> tuple[YoloV1, DatasetInfo, dict]:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    info = DatasetInfo(**ckpt["info"])
    model = build_model(ModelConfig(**ckpt["configs"]["model"]), info)
    model.load_state_dict(ckpt["model"])
    return model.to(device).eval(), info, ckpt["configs"]


def train(
    data_cfg: DataConfig,
    model_cfg: ModelConfig,
    cfg: TrainConfig,
    resume: str | None = None,
    callback: Callable[[dict], None] | None = None,
) -> dict:
    """Train and return the best validation metrics. `callback(event)` is called after every epoch."""
    seed_everything(cfg.seed)
    device = resolve_device(cfg.device)
    out = Path(cfg.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    configs = {
        "data": dataclasses.asdict(data_cfg),
        "model": dataclasses.asdict(model_cfg),
        "train": dataclasses.asdict(cfg),
    }

    datasets = build_datasets(data_cfg, overfit=cfg.overfit)
    info = DatasetInfo(datasets.class_names, data_cfg.image_size, data_cfg.grid_size, data_cfg.num_boxes)
    train_loader, val_loader = build_dataloaders(datasets, cfg, data_cfg, device)
    model = build_model(model_cfg, info).to(device)
    loss_fn = YoloLoss(info.grid_size, info.num_boxes, info.num_classes, cfg.lambda_coord, cfg.lambda_noobj)
    log.info(
        "%s: %d classes, train %d, val %d | model %s x%.2f, %.1fM parameters | %s",
        data_cfg.dataset, info.num_classes, len(datasets.train), len(datasets.val),
        model_cfg.arch, model_cfg.width, model.num_parameters() / 1e6, device,
    )  # fmt: skip

    decay, no_decay = [], []
    for p in model.parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.weight_decay}, {"params": no_decay, "weight_decay": 0.0}], lr=cfg.lr
    )
    total_steps = cfg.epochs * len(train_loader)
    warmup_steps = max(1, int(cfg.warmup_epochs * len(train_loader)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: lr_lambda(s, warmup_steps, total_steps))
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history: list[dict] = []
    start_epoch, best_map = 1, -1.0
    if resume:
        ckpt = torch.load(resume, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch, best_map = ckpt["epoch"] + 1, ckpt["best_map"]
        history_path = Path(resume).with_name("history.json")
        if history_path.exists():
            history = json.loads(history_path.read_text())
        log.info("resumed from %s at epoch %d (best mAP %.3f)", resume, ckpt["epoch"], best_map)

    best = {}
    for epoch in range(start_epoch, cfg.epochs + 1):
        model.train()
        t0, running, components, n = time.time(), 0.0, {"coord": 0.0, "obj": 0.0, "noobj": 0.0, "class": 0.0}, 0
        for step, (pixels, grid, _) in enumerate(train_loader, 1):
            pixels, grid = pixels.to(device, non_blocking=True), grid.to(device, non_blocking=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                output = model(pixels)
            loss = loss_fn(output, grid)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
            for k in components:
                components[k] += loss_fn.last_components[k]
            n += 1
            if cfg.log_every and step % cfg.log_every == 0:
                log.info("epoch %d step %d/%d loss %.3f", epoch, step, len(train_loader), running / n)

        val = evaluate(model, val_loader, loss_fn, info, cfg, device)
        record = {
            "epoch": epoch,
            "train_loss": running / max(1, n),
            **{f"train_{k}": v / max(1, n) for k, v in components.items()},
            "val_loss": val["loss"],
            "val_map": val["map"],
            "lr": scheduler.get_last_lr()[0],
            "seconds": time.time() - t0,
        }
        history.append(record)
        log.info(
            "epoch %d/%d  train loss %.3f  val loss %.3f  val mAP@%.1f %.3f  (%.0fs)",
            epoch, cfg.epochs, record["train_loss"], val["loss"], cfg.map_iou, val["map"], record["seconds"],
        )  # fmt: skip
        if val["map"] > best_map:
            best_map = val["map"]
            best = {"epoch": epoch, "map": val["map"], "ap": val["ap"], "val_loss": val["loss"]}
            save_checkpoint(out / BEST_CHECKPOINT, model, optimizer, scheduler, epoch, best_map, configs, info)
        save_checkpoint(out / LAST_CHECKPOINT, model, optimizer, scheduler, epoch, best_map, configs, info)
        (out / "history.json").write_text(json.dumps(history, indent=2))
        if epoch == cfg.epochs or (cfg.plot_every and epoch % cfg.plot_every == 0):
            plot_history(history, out / "history.png")
            ex = val["examples"]
            if ex["images"]:
                plot_predictions(
                    ex["images"], ex["predictions"], ex["targets"], info.class_names, out / "predictions.png"
                )
        if callback:
            callback({**record, "ap": val["ap"], "best_map": best_map})

    (out / "best_metrics.json").write_text(json.dumps(best, indent=2))
    log.info("best validation mAP@%.1f %.3f at epoch %s", cfg.map_iou, best_map, best.get("epoch"))
    return best


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Train YOLOv1.", formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    for group_name, config_cls in (("data", DataConfig), ("model", ModelConfig), ("training", TrainConfig)):
        group = parser.add_argument_group(group_name)
        for f in fields(config_cls):
            flag = "--" + f.name.replace("_", "-")
            if group_name == "training" and f.name == "seed":
                continue  # one --seed flag (data group) feeds both configs
            if isinstance(f.default, bool):
                group.add_argument(flag, action=argparse.BooleanOptionalAction, default=f.default)
            elif f.name == "classes":
                group.add_argument(flag, nargs="+", default=None, help="class names (yolo dataset)")
            else:
                group.add_argument(flag, type=type(f.default), default=f.default)
    parser.add_argument("--resume", default=None, help="continue from a last.pt checkpoint")
    return parser


def configs_from_args(args: argparse.Namespace) -> tuple[DataConfig, ModelConfig, TrainConfig]:
    def pick(config_cls):
        return config_cls(**{f.name: getattr(args, f.name) for f in fields(config_cls) if hasattr(args, f.name)})

    data_cfg, model_cfg, train_cfg = pick(DataConfig), pick(ModelConfig), pick(TrainConfig)
    train_cfg.seed = data_cfg.seed
    return data_cfg, model_cfg, train_cfg


def main(argv: list[str] | None = None) -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
    args = build_parser().parse_args(argv)
    train(*configs_from_args(args), resume=args.resume)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/yolov1/evaluate.py
"""The `yolov1-evaluate` command: mAP of a checkpoint on the validation split of a dataset."""

from __future__ import annotations

import argparse
import json
import logging
from pathlib import Path

import torch

from yolov1.config import DataConfig, TrainConfig
from yolov1.data import build_datasets
from yolov1.loss import YoloLoss
from yolov1.train import build_dataloaders, evaluate, load_model, resolve_device
from yolov1.viz import plot_predictions


def main(argv: list[str] | None = None) -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
    parser = argparse.ArgumentParser(description="Evaluate a YOLOv1 checkpoint (mAP@IoU on the validation split).")
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--root", default=None, help="dataset folder (defaults to the one used for training)")
    parser.add_argument("--conf-threshold", type=float, default=0.1)
    parser.add_argument("--nms-iou", type=float, default=0.5)
    parser.add_argument("--map-iou", type=float, default=0.5)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--report", default=None, help="write the metrics to this json file")
    parser.add_argument("--predictions", default=None, help="write a prediction grid to this png file")
    parser.add_argument("--device", default="auto")
    args = parser.parse_args(argv)

    device = resolve_device(args.device)
    model, info, configs = load_model(args.checkpoint, device)
    data_cfg = DataConfig(**configs["data"])
    if args.root:
        data_cfg.root = args.root
    cfg = TrainConfig(
        batch_size=args.batch_size, conf_threshold=args.conf_threshold, nms_iou=args.nms_iou, map_iou=args.map_iou
    )
    datasets = build_datasets(data_cfg)
    _, val_loader = build_dataloaders(datasets, cfg, data_cfg, device)
    loss_fn = YoloLoss(info.grid_size, info.num_boxes, info.num_classes)
    with torch.no_grad():
        result = evaluate(model, val_loader, loss_fn, info, cfg, device)
    print(
        f"validation images: {len(datasets.val)}   loss {result['loss']:.3f}   mAP@{args.map_iou} {result['map']:.4f}"
    )
    for name, ap in sorted(result["ap"].items(), key=lambda kv: -kv[1]):
        print(f"  {name:24} AP {ap:.4f}")
    if args.report:
        Path(args.report).parent.mkdir(parents=True, exist_ok=True)
        report = {"map": result["map"], "map_iou": args.map_iou, "loss": result["loss"], "ap": result["ap"]}
        Path(args.report).write_text(json.dumps(report, indent=2))
    if args.predictions:
        ex = result["examples"]
        plot_predictions(ex["images"], ex["predictions"], ex["targets"], info.class_names, args.predictions)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/yolov1/predict.py
"""The `yolov1-predict` command: detect objects in image files or a webcam stream with a trained checkpoint."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch
from PIL import Image

from yolov1.data.transforms import eval_transforms
from yolov1.train import load_model, postprocess
from yolov1.viz import draw_boxes


class Detector:
    def __init__(self, checkpoint: str | Path, device: str = "cpu", conf_threshold: float = 0.3, nms_iou: float = 0.5):
        self.model, self.info, self.configs = load_model(checkpoint, device)
        self.device, self.conf_threshold, self.nms_iou = torch.device(device), conf_threshold, nms_iou
        self.transforms = eval_transforms(self.info.image_size)
        self.class_names = self.info.class_names

    @torch.no_grad()
    def detect(self, image: Image.Image) -> torch.Tensor:
        """PIL image -> (K, 6) [class, score, cx, cy, w, h], coordinates relative to the original image."""
        pixels, _ = self.transforms(image.convert("RGB"), torch.zeros(0, 5))
        output = self.model(pixels.unsqueeze(0).to(self.device))
        return postprocess(output, self.info, self.conf_threshold, self.nms_iou)[0]

    def to_records(self, detections: torch.Tensor, width: int, height: int) -> list[dict]:
        records = []
        for cls, score, cx, cy, w, h in detections.tolist():
            records.append(
                {
                    "class": self.class_names[int(cls)],
                    "score": round(score, 4),
                    "box_xyxy": [
                        round((cx - w / 2) * width, 1),
                        round((cy - h / 2) * height, 1),
                        round((cx + w / 2) * width, 1),
                        round((cy + h / 2) * height, 1),
                    ],
                }
            )
        return records


def run_camera(detector: Detector, camera: int = 0) -> None:
    import cv2  # optional dependency
    import numpy as np

    cap = cv2.VideoCapture(camera)
    if not cap.isOpened():
        raise SystemExit(f"could not open camera {camera}")
    print("press q to quit")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        drawn = draw_boxes(image, detector.detect(image), detector.class_names)
        cv2.imshow("YOLOv1", cv2.cvtColor(np.array(drawn), cv2.COLOR_RGB2BGR))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description="Detect objects with a trained YOLOv1 checkpoint.")
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--images", nargs="*", default=[], help="image files")
    parser.add_argument("--camera", type=int, default=None, help="webcam index for live detection (needs opencv)")
    parser.add_argument("--output-dir", default="outputs")
    parser.add_argument("--conf-threshold", type=float, default=0.3)
    parser.add_argument("--nms-iou", type=float, default=0.5)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args(argv)

    detector = Detector(args.checkpoint, args.device, args.conf_threshold, args.nms_iou)
    if args.camera is not None:
        run_camera(detector, args.camera)
        return
    if not args.images:
        parser.error("give --images or --camera")
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    results = {}
    for path in args.images:
        image = Image.open(path).convert("RGB")
        detections = detector.detect(image)
        draw_boxes(image, detections, detector.class_names).save(out / f"{Path(path).stem}_pred.jpg")
        results[path] = detector.to_records(detections, *image.size)
        found = ", ".join(f"{r['class']} {r['score']:.2f}" for r in results[path])
        print(f"{path}: {len(detections)} objects {found}")
    (out / "detections.json").write_text(json.dumps(results, indent=2))
    print(f"wrote {out}/detections.json and {len(results)} annotated images")


if __name__ == "__main__":
    main()

In [ ]:
import sys

sys.path.insert(0, "src")
for name in [m for m in sys.modules if m.startswith("yolov1")]:
    del sys.modules[name]  # pick up the files just written if this cell is re-run
import yolov1.train as T

print("yolov1 ready")

## 2. Settings

| `DATASET` | classes | images | download | what for |
|---|---|---|---|---|
| `"shapes"` | 3 | generated, exact boxes | none | prove the detector learns (minutes) |
| `"mnist"` | 10 | MNIST digits pasted on a canvas | 12 MB | real pixels, exact boxes |
| `"voc"` | 20 | PASCAL VOC 2012 | ~2 GB | the benchmark of the paper (hours) |
| `"yolo"` | yours | any YOLO text-format folder | your data | e.g. Mapillary traffic signs (set `YOLO_ROOT`) |

`SMOKE_TEST = True` runs a small version to check the whole notebook in a few minutes. Keep it
`False` for real training.

In [ ]:
import dataclasses
from pathlib import Path

import torch

from yolov1.config import DataConfig, ModelConfig, TrainConfig

SMOKE_TEST = False          # True = small quick check of the whole notebook
DATASET = "voc"             # "shapes" | "mnist" | "voc" | "yolo"
USE_GOOGLE_DRIVE = False    # Colab only: keep checkpoints on Drive (survive disconnects)
RESUME = True               # continue from last.pt in OUTPUT_DIR if it exists

# "yolo" dataset: a folder with images/ + labels/ (+ classes.txt), or train/ val/ sub-folders.
# Kaggle: add the dataset to the notebook and point YOLO_ROOT at /kaggle/input/<dataset>.
YOLO_ROOT = "/kaggle/input/traffic-signs-dataset-mapillary-and-dfg"
IMAGES_DIR, LABELS_DIR = "images", "txts (YOLO)"
YOLO_CLASSES = None         # list of class names, or None to read classes.txt / infer the ids

# --- where checkpoints go --------------------------------------------------------
ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path(f"/content/drive/MyDrive/yolo-v1-object-detector/{DATASET}")
elif ON_KAGGLE:
    OUTPUT_DIR = Path(f"/kaggle/working/checkpoints/{DATASET}")
else:
    OUTPUT_DIR = Path(f"checkpoints/{DATASET}")

# --- per-dataset presets -----------------------------------------------------------
PRESETS = {
    "shapes": dict(data=dict(image_size=112, train_size=4000, val_size=400),
                   model=dict(arch="small", hidden=512), epochs=25, batch_size=64, lr=1e-3),
    "mnist": dict(data=dict(image_size=112, train_size=20000, val_size=1000, root="data", augment=False),
                  model=dict(arch="small", hidden=512, dropout=0.3), epochs=15, batch_size=64, lr=1e-3),
    "voc": dict(data=dict(image_size=448, root="data"),
                model=dict(arch="yolov1", hidden=4096), epochs=135, batch_size=32, lr=1e-4),
    "yolo": dict(data=dict(image_size=448, root=YOLO_ROOT, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
                           classes=YOLO_CLASSES),
                 model=dict(arch="yolov1", hidden=4096), epochs=100, batch_size=32, lr=1e-4),
}
preset = PRESETS[DATASET]
data_config = DataConfig(dataset=DATASET, num_workers=min(4, os.cpu_count() or 1), **preset["data"])
model_config = ModelConfig(**preset["model"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.0f} GB)")
else:
    print("No GPU: enable one in the notebook settings (fine only for SMOKE_TEST / shapes).")

train_config = TrainConfig(
    output_dir=str(OUTPUT_DIR), epochs=preset["epochs"], batch_size=preset["batch_size"], lr=preset["lr"],
    warmup_epochs=2, device=device.type, log_every=25, plot_every=0,
)
if SMOKE_TEST:
    OUTPUT_DIR = OUTPUT_DIR.parent / "smoke-test"  # never mixed with a real run
    data_config = dataclasses.replace(data_config, train_size=64, val_size=16)
    model_config = dataclasses.replace(model_config, width=0.25, hidden=128)
    train_config = dataclasses.replace(
        train_config, output_dir=str(OUTPUT_DIR), epochs=3, batch_size=8, warmup_epochs=1, log_every=2,
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
resume = str(OUTPUT_DIR / T.LAST_CHECKPOINT) if RESUME and (OUTPUT_DIR / T.LAST_CHECKPOINT).exists() else None

for name, cfg in [("DataConfig", data_config), ("ModelConfig", model_config), ("TrainConfig", train_config)]:
    print(f"\n{name}:")
    for key, value in dataclasses.asdict(cfg).items():
        print(f"  {key:16} {value}")

## 3. Data

A look at a few training images with their ground-truth boxes (after augmentation). VOC and
MNIST are downloaded here the first time.

In [ ]:
import logging

import matplotlib.pyplot as plt

from yolov1.data import build_datasets
from yolov1.viz import draw_boxes, tensor_to_pil

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
T.seed_everything(train_config.seed)
datasets = build_datasets(data_config)
class_names = datasets.class_names
print(f"{len(class_names)} classes | train {len(datasets.train):,} | val {len(datasets.val):,}")

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
for ax, i in zip(axes, range(0, len(datasets.train), max(1, len(datasets.train) // 6))):
    pixels, _, boxes = datasets.train[i]
    ax.imshow(draw_boxes(tensor_to_pil(pixels), boxes, class_names, with_scores=False))
    ax.axis("off")
plt.show()

## 4. Model

In [ ]:
info = T.DatasetInfo(class_names, data_config.image_size, data_config.grid_size, data_config.num_boxes)
model = T.build_model(model_config, info).to(device)
print(f"{model_config.arch} x{model_config.width}: {model.num_parameters() / 1e6:.1f}M parameters, "
      f"stride {model.stride}, output {info.grid_size}x{info.grid_size}x{info.num_classes + 5 * info.num_boxes}")

## 5. Training with a live dashboard

Same loop as `yolov1-train`: AdamW, warmup + cosine schedule, mixed precision, gradient
clipping, the paper's loss. After every epoch it evaluates mAP@0.5 on the validation split and
saves `last.pt`, plus `best.pt` when the mAP improves.

**Interrupted, or hit the session limit?** Run the notebook again with `RESUME = True`: it
continues from `last.pt`.

In [ ]:
from IPython.display import clear_output

history = []


def draw_dashboard(event):
    history.append(event)
    clear_output(wait=True)
    epochs = [h["epoch"] for h in history]
    fig, (ax_loss, ax_map, ax_parts) = plt.subplots(1, 3, figsize=(16, 4))
    ax_loss.plot(epochs, [h["train_loss"] for h in history], label="train")
    ax_loss.plot(epochs, [h["val_loss"] for h in history], "o-", label="validation")
    ax_loss.set_title("YOLO loss")
    ax_map.plot(epochs, [h["val_map"] for h in history], "o-", color="tab:green")
    ax_map.set_title("validation mAP@0.5")
    ax_map.set_ylim(0, 1)
    for key in ("coord", "obj", "noobj", "class"):
        ax_parts.plot(epochs, [h["train_" + key] for h in history], label=key)
    ax_parts.set_title("train loss components")
    for ax in (ax_loss, ax_map, ax_parts):
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend()
    plt.show()
    print(f"epoch {event['epoch']}/{train_config.epochs} | train loss {event['train_loss']:.3f} | "
          f"val loss {event['val_loss']:.3f} | val mAP@0.5 {event['val_map']:.3f} "
          f"(best {event['best_map']:.3f}) | {event['seconds']:.0f}s")


best = T.train(data_config, model_config, train_config, resume=resume, callback=draw_dashboard)
print(f"Training finished. Best validation mAP@0.5 {best['map']:.3f} at epoch {best['epoch']}")

## 6. Results

The best checkpoint on the validation split: mAP, per-class AP and a grid of predictions.

In [ ]:
from IPython.display import Image as IPImage, display

from yolov1 import evaluate

evaluate.main(["--checkpoint", str(OUTPUT_DIR / T.BEST_CHECKPOINT), "--report", str(OUTPUT_DIR / "report.json"),
               "--predictions", str(OUTPUT_DIR / "predictions.png"), "--device", device.type])
display(IPImage(str(OUTPUT_DIR / "predictions.png")))
display(IPImage(str(OUTPUT_DIR / "history.png")))

## 7. Download the trained model

Zips `best.pt`, the report and the figures (`last.pt` is left out: it is only needed to
resume). Use it with the repo:

```bash
unzip yolov1-run.zip -d checkpoints/run
yolov1-predict --checkpoint checkpoints/run/best.pt --images street.jpg
yolov1-predict --checkpoint checkpoints/run/best.pt --camera 0
```

In [ ]:
import shutil
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    bundle = Path(tmp) / "bundle"
    bundle.mkdir()
    keep = [T.BEST_CHECKPOINT, "report.json", "history.json", "history.png", "predictions.png", "best_metrics.json"]
    for name in keep:
        if (OUTPUT_DIR / name).exists():
            shutil.copy(OUTPUT_DIR / name, bundle / name)
    archive = shutil.make_archive(str(OUTPUT_DIR.parent / "yolov1-run"), "zip", bundle)
print("created", archive)

if ON_COLAB:
    from google.colab import files

    files.download(archive)
elif ON_KAGGLE:
    print("Kaggle: click 'Save Version', then download it from the notebook's Output tab.")